# preQCA — Preparar los datos para fsQCA

QCA necesita representar las condiciones y el resultado como **pertenencias a conjuntos**.

En este notebook partiremos de `base_completa`, que contiene las métricas de las redes de contratación y el INCO de los gobiernos regionales para 2021, 2022 y 2024. Conoceremos las mediciones originales, definiremos los conjuntos, estableceremos y justificaremos los umbrales de calibración, transformaremos las mediciones en membresías, compararemos ambos valores y guardaremos la base preparada.

Las condiciones representarán características de las relaciones de contratación de los gobiernos regionales con las empresas. El resultado será la pertenencia al conjunto de gobiernos regionales con INCO alto.

**Unidad de análisis:** cada gobierno regional en un año determinado (GORE–año).

`mediciones originales → conceptualización → umbrales → calibración → membresías`

In [1]:
!pip -q install setqca

## preQCA-01: conocer las mediciones originales

Cada fila representa un **gobierno regional en un año determinado (GORE–año)**.

Las métricas de las redes de contratación y el INCO todavía son **mediciones empíricas**, no membresías QCA.

Antes de calibrarlas, precisaremos para cada indicador su definición, importancia teórica, fuente, rango, referencia y temporalidad.

In [3]:
import pandas as pd

linkGit = "https://raw.githubusercontent.com/romloayza/networks_x_qca_tesis/refs/heads/main/data/base_completa.csv"

qca_raw = pd.read_csv(linkGit)

qca_raw.head()

,GORE,AÑO,n_contratos,monto_total,grado,grado_ponderado,centralidad_grado,centralidad_intermediacion,centralidad_cercania_armonica,centralidad_eigenvector,...,tasa_recurrencia,pct_contratos_consorcio,empresas_por_contrato,emp_n_aristas,emp_densidad,emp_grado_promedio,emp_clustering,emp_n_componentes,emp_pct_componente_mayor,INCO
0,AMAZONAS,2021,21,6.434618e+07,28,38,0.076923,0.239361,0.288755,0.029060,...,0.464286,0.809524,1.809524,16,0.042328,1.142857,0.000000,12.0,0.142857,68.3
1,AMAZONAS,2022,16,3.100791e+08,23,27,0.041367,0.069163,0.194006,0.000109,...,0.391304,0.687500,1.687500,11,0.043478,0.956522,0.000000,12.0,0.173913,67.2
2,AMAZONAS,2024,27,7.218695e+07,37,48,0.055389,0.098990,0.296441,0.010163,...,0.432432,0.777778,1.777778,19,0.028529,1.027027,0.000000,18.0,0.135135,49.0
3,ANCASH,2021,52,2.086562e+08,77,103,0.211538,0.504791,0.388341,0.705884,...,0.467532,0.865385,1.980769,58,0.019822,1.506494,0.173160,28.0,0.103896,88.9
4,ANCASH,2022,103,5.040132e+08,142,199,0.255396,0.663705,0.457357,0.705560,...,0.408451,0.883495,1.932039,95,0.009490,1.338028,0.079108,54.0,0.154930,84.1


In [4]:
qca_raw = qca_raw[
    [
        "GORE",
        "AÑO",
        "centralidad_grado",
        "pct_empresas_compartidas",
        "hhi_contratos",
        "centralidad_intermediacion",
        "INCO"
    ]
].copy()

qca_raw.head()

,GORE,AÑO,centralidad_grado,pct_empresas_compartidas,hhi_contratos,centralidad_intermediacion,INCO
0,AMAZONAS,2021,0.076923,0.178571,0.044321,0.239361,68.3
1,AMAZONAS,2022,0.041367,0.086957,0.050754,0.069163,67.2
2,AMAZONAS,2024,0.055389,0.162162,0.061632,0.098990,49.0
3,ANCASH,2021,0.211538,0.090909,0.017438,0.504791,88.9
4,ANCASH,2022,0.255396,0.112676,0.010025,0.663705,84.1


## De las mediciones a los conjuntos

A partir de las mediciones originales construiremos cinco conjuntos: cuatro condiciones y un resultado.

- `ALCANCE`: GORE–año con alta centralidad de grado en la red bipartita de contratación.
- `COMPARTIDAS`: GORE–año con alta proporción de empresas contratistas compartidas con otros gobiernos regionales.
- `CONCENTRACION`: GORE–año con alta concentración de participaciones contractuales entre sus empresas contratistas, medida mediante el HHI de contratos.
- `INTERMEDIACION`: GORE–año con alta centralidad de intermediación en la red bipartita de contratación.
- `INCO_ALTO`: GORE–año con alto puntaje de riesgo de corrupción e inconducta funcional según el INCO.

Los cuatro primeros conjuntos representan las condiciones y el último representa el resultado.

La medición original y el conjunto no son lo mismo. Aunque las métricas de red estén expresadas entre 0 y 1, sus valores no constituyen automáticamente membresías QCA. La calibración permitirá traducir cada medición empírica en un grado de pertenencia entre 0 y 1, utilizando umbrales sustantivamente justificados.

## preQCA-02: establecer los umbrales

Utilizaremos los siguientes umbrales como propuesta inicial de calibración. Son decisiones provisionales de la investigación, no cortes teóricos universales ni categorías oficiales.

| Conjunto | Fuera | Cruce | Dentro |
|---|---:|---:|---:|
| ALCANCE | 0.01 | 0.04 | 0.10 |
| COMPARTIDAS | 0.10 | 0.25 | 0.50 |
| CONCENTRACION | 0.025 | 0.10 | 0.25 |
| INTERMEDIACION | 0.01 | 0.10 | 0.25 |
| INCO_ALTO | 30 | 50 | 70 |

Estos valores definen cómo interpretaremos cada medición para construir su conjunto. Su justificación se revisará mediante referencias externas y conocimiento de los casos, y evaluaremos la sensibilidad de los resultados a calibraciones alternativas. No se elegirán para obtener determinadas configuraciones QCA.

In [5]:
umbrales = {
    "ALCANCE": (0.01, 0.04, 0.10),
    "COMPARTIDAS": (0.10, 0.25, 0.50),
    "CONCENTRACION": (0.025, 0.10, 0.25),
    "INTERMEDIACION": (0.01, 0.10, 0.25),
    "INCO_ALTO": (30, 50, 70)
}

umbrales

{'ALCANCE': (0.01, 0.04, 0.1),
 'COMPARTIDAS': (0.1, 0.25, 0.5),
 'CONCENTRACION': (0.025, 0.1, 0.25),
 'INTERMEDIACION': (0.01, 0.1, 0.25),
 'INCO_ALTO': (30, 50, 70)}

## preQCA-03: calibrar las mediciones

`calibrate_direct()` recibe una medición y tres referencias sustantivas:

- `full_out`: anclaje de plena no pertenencia al conjunto;
- `crossover`: punto de máxima ambigüedad;
- `full_in`: anclaje de plena pertenencia al conjunto.

Calibraremos por separado las cuatro métricas de red y el INCO, utilizando los umbrales definidos para cada conjunto:

- `centralidad_grado` → `ALCANCE`;
- `pct_empresas_compartidas` → `COMPARTIDAS`;
- `hhi_contratos` → `CONCENTRACION`;
- `centralidad_intermediacion` → `INTERMEDIACION`;
- `INCO` → `INCO_ALTO`.

Conservaremos las mediciones originales y crearemos nuevas columnas con las membresías entre 0 y 1. Los valores ausentes permanecerán como tales y no se reemplazarán por cero.

In [8]:
from setqca import calibrate_direct

qca_datos = qca_raw.dropna(subset=[
    "centralidad_grado",
    "pct_empresas_compartidas",
    "hhi_contratos",
    "centralidad_intermediacion",
    "INCO"
]).copy()

qca = pd.DataFrame({
    "caso": (
        qca_datos["GORE"] + "_" + qca_datos["AÑO"].astype(str)
    ),

    "ALCANCE": calibrate_direct(
        qca_datos["centralidad_grado"],
        full_out=0.01,
        crossover=0.04,
        full_in=0.10
    ),

    "COMPARTIDAS": calibrate_direct(
        qca_datos["pct_empresas_compartidas"],
        full_out=0.10,
        crossover=0.25,
        full_in=0.50
    ),

    "CONCENTRACION": calibrate_direct(
        qca_datos["hhi_contratos"],
        full_out=0.025,
        crossover=0.10,
        full_in=0.25
    ),

    "INTERMEDIACION": calibrate_direct(
        qca_datos["centralidad_intermediacion"],
        full_out=0.01,
        crossover=0.10,
        full_in=0.25
    ),

    "INCO_ALTO": calibrate_direct(
        qca_datos["INCO"],
        full_out=30,
        crossover=50,
        full_in=70
    )
})

qca.head()

,caso,ALCANCE,COMPARTIDAS,CONCENTRACION,INTERMEDIACION,INCO_ALTO
0,AMAZONAS_2021,0.859599,0.197481,0.101024,0.939095,0.936681
1,AMAZONAS_2022,0.516764,0.039148,0.126381,0.267202,0.926370
2,AMAZONAS_2024,0.680313,0.151328,0.181488,0.491741,0.463261
3,ANCASH_2021,0.999779,0.042173,0.037640,0.999646,0.996754
4,ANCASH_2022,0.999974,0.063233,0.028406,0.999984,0.993441


## preQCA-04: comparar mediciones y membresías

Ahora observamos lado a lado las **mediciones originales** y las **membresías calibradas**.

`medición original → pertenencia al conjunto`


In [9]:
comparacion = pd.DataFrame({
    "caso": qca["caso"],
    "centralidad_grado": qca_datos["centralidad_grado"],
    "ALCANCE": qca["ALCANCE"],
    "pct_empresas_compartidas": qca_datos["pct_empresas_compartidas"],
    "COMPARTIDAS": qca["COMPARTIDAS"],
    "hhi_contratos": qca_datos["hhi_contratos"],
    "CONCENTRACION": qca["CONCENTRACION"],
    "centralidad_intermediacion": qca_datos["centralidad_intermediacion"],
    "INTERMEDIACION": qca["INTERMEDIACION"],
    "INCO": qca_datos["INCO"],
    "INCO_ALTO": qca["INCO_ALTO"]
})

comparacion.head(10)

,caso,centralidad_grado,ALCANCE,pct_empresas_compartidas,COMPARTIDAS,hhi_contratos,CONCENTRACION,centralidad_intermediacion,INTERMEDIACION,INCO,INCO_ALTO
0,AMAZONAS_2021,0.076923,0.859599,0.178571,0.197481,0.044321,0.101024,0.239361,0.939095,68.3,0.936681
1,AMAZONAS_2022,0.041367,0.516764,0.086957,0.039148,0.050754,0.126381,0.069163,0.267202,67.2,0.926370
2,AMAZONAS_2024,0.055389,0.680313,0.162162,0.151328,0.061632,0.181488,0.098990,0.491741,49.0,0.463261
3,ANCASH_2021,0.211538,0.999779,0.090909,0.042173,0.017438,0.037640,0.504791,0.999646,88.9,0.996754
4,ANCASH_2022,0.255396,0.999974,0.112676,0.063233,0.010025,0.028406,0.663705,0.999984,84.1,0.993441
5,ANCASH_2024,0.007485,0.039495,0.600000,0.984050,0.200000,0.876853,0.008402,0.047574,67.9,0.933097
6,APURIMAC_2021,0.005495,0.032715,0.500000,0.950000,0.500000,0.999611,0.000081,0.036652,79.5,0.987170
7,APURIMAC_2022,0.003597,0.027309,0.000000,0.007337,0.500000,0.999611,0.000006,0.036565,74.4,0.973201
9,AREQUIPA_2021,0.019231,0.115224,0.285714,0.603635,0.142857,0.698732,0.023498,0.075659,94.1,0.998488
10,AREQUIPA_2022,0.016187,0.088089,0.111111,0.061437,0.120000,0.596906,0.026657,0.083213,70.1,0.950695


In [10]:
qca.to_csv("qca_calibrada.csv", index=False)

print("Archivo creado: qca_calibrada.csv")
print("Casos:", len(qca))


Archivo creado: qca_calibrada.csv
Casos: 69


In [11]:
from google.colab import files

files.download("qca_calibrada.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>